In [ ]:
%load_ext autoreload
%autoreload 2
import os
print(os.getcwd())

/home/as/code/ai/susteelaible/nlp


# bert-1
- run pipeline

In [ ]:
from nlp import ClimateBERTAnalyzer, analyze_reports

stats = analyze_reports('../data/reports/Baosteel')
# stats = analyze_reports("../data/reports")

# bert-2
- run vizualisations

In [ ]:
from nlp import ClimateBERTVisualizer, visualize_results

visualize_results("../cache", "../out")

✅ Loaded: 15 reports, 1 companies, 2013-2020

EXPORTING CSV FILES
   ✓ company_year.csv (8 rows)
   ✓ company_totals.csv (1 companies)
   ✓ yearly_industry.csv (8 years)
   ✓ funnel_company_year.csv (8 rows)

   📁 All CSVs saved to: ../out/

GENERATING PLOTS
   ✓ slide_main.png
   ✓ slide_sentiment_trend.png
   ✓ talk_score_trend.png
   ✓ funnel_trend.png
   ✓ talk_score_per_company.png
   ✓ per_company_components.png
   ✓ per_company_sentiment.png
   ✓ sentiment_all_companies.png
   ✓ n0_funnel.png
   ✓ n0_quality_comparison.png
   ✓ n0_per_company.png
   ✓ n0_gap_analysis.png

   📁 All plots saved to: ../out/

   Generating word frequency plots...

   📊 ALL CHUNKS (top 30):
   environment(1232), iron(766), management(736), development(720), energy(628), technology(609), products(607), protection(559), production(525), industry(522), green(473), emission(453), system(428), reduction(421), base(410)

   🌱 OPPORTUNITY chunks (top 15):
   environment(809), development(574), technology(52

# rag 1

## LLM Model & Parameter Selection

Two-step process to find optimal LLM settings for your GPU:

1. **Model Selection** - Find which model is fast + follows format
2. **Parameter Tuning** - Optimize ctx window and batch size

### Hardware Context
- T1200 4GB VRAM: Models >3GB will partially offload to CPU (slower)
- KV cache needs VRAM too: larger context = more memory

### Model Notes (tested on T1200 4GB)
| Model | Size | Speed | Format | Notes |
|-------|------|-------|--------|-------|
| `gemma3:1b` | 1.5GB | 2.8s | ✅ | Fast, fits fully, but quality degrades with >7 chunks |
| `gemma3:4b` | 3.3GB | 8.8s | ✅ | Good quality but 45/55 CPU/GPU split |
| `llama3.2:3b` | 2GB | 0.5s | ❌ | Returns NONE_FOUND, doesn't understand task |
| `phi3:mini` | 3.8GB | 3.4s | ❌ | Mangles chunk IDs |
| `qwen3:4b` | 4.4GB | 108s | ✅ | Hidden "thinking" tokens, extremely slow |

In [10]:
# =============================================================================
# TEST 1: MODEL SELECTION - Find which model works on your GPU
# =============================================================================
# Run this first to find a model that is: (1) fast, (2) follows output format
# Unloads each model after testing to ensure fair comparison

import time
import subprocess
from langchain_ollama import ChatOllama

# Test prompts: small (baseline) and realistic (simulates actual extraction)
SMALL_PROMPT = """Extract BARRIERS. Format: [chunk_id]|||text

[012_001]
The high cost of green hydrogen limits decarbonisation options.

[012_002]
We committed to reducing emissions by 50% by 2030."""

REALISTIC_PROMPT = """Extract BARRIERS to decarbonisation from this text.
OUTPUT FORMAT: [chunk_id]|||verbatim text
If none: NONE_FOUND

TEXT:
[012_001]
The high cost of green hydrogen remains a significant barrier to decarbonisation in the steel industry. Current production costs for green hydrogen are approximately 4-6 times higher than grey hydrogen produced from natural gas. This cost differential makes it economically challenging for steel producers to transition their operations.

[012_002]
Infrastructure limitations present another major challenge. The existing natural gas pipeline network cannot be directly repurposed for hydrogen transport due to material compatibility issues. Building dedicated hydrogen infrastructure requires substantial capital investment.

[012_003]
Technological readiness varies significantly across different decarbonisation pathways. While electric arc furnace technology is mature, it requires high-quality scrap steel that is increasingly scarce. Direct reduced iron technology using hydrogen is still being piloted at scale.

[012_004]
Regulatory uncertainty continues to hamper investment decisions. The EU Emissions Trading System has seen significant price volatility, making it difficult for companies to build business cases for low-carbon investments.

[012_005]
Skills gaps and workforce transition present social challenges. The shift to new production technologies requires retraining of existing workers and recruitment of specialists in hydrogen systems and digital process control."""

def unload_model(model):
    """Unload model from VRAM between tests."""
    try:
        subprocess.run(["ollama", "stop", model], capture_output=True, timeout=10)
        time.sleep(2)
    except:
        pass

def test_model(model, num_ctx=4096):
    """Test model with both small and realistic prompts."""
    llm = ChatOllama(model=model, temperature=0, num_ctx=num_ctx)

    print(f"\n{model} (ctx={num_ctx}):")

    # Warm-up (loads model into VRAM)
    start = time.time()
    llm.invoke("Hi")
    print(f"  Warm-up: {time.time() - start:.1f}s")

    # Small prompt test
    start = time.time()
    resp_small = llm.invoke(SMALL_PROMPT)
    time_small = time.time() - start

    # Realistic prompt test (5 chunks, ~1200 tokens)
    start = time.time()
    resp_real = llm.invoke(REALISTIC_PROMPT)
    time_real = time.time() - start

    # Check format compliance
    correct = "[012_00" in resp_real.content and "|||" in resp_real.content

    print(f"  Small prompt:     {time_small:.2f}s")
    print(f"  Realistic prompt: {time_real:.2f}s ({'✅' if correct else '❌'} format)")
    print(f"  Output: {resp_real.content[:120]}...")

    unload_model(model)
    return time_small, time_real, correct

# === MODELS TO TEST ===
# For 4GB VRAM: need model <3GB to leave room for KV cache
models_to_test = [
    ("qwen2.5:3b", 4096),   # 1.9GB - NO thinking mode (unlike qwen3)!
    # ("gemma3:1b", 4096),    # 815MB - fast but quality issues with >7 chunks
    # ("gemma3:4b", 4096),    # 3.3GB - good quality but 50% CPU offload on 4GB VRAM
    # ("llama3.2:3b", 4096),  # 2.0GB - returns NONE_FOUND, doesn't understand task
    # ("phi3:mini", 4096),    # 2.2GB - fast but mangles chunk IDs
    # ("qwen3:4b", 4096),     # 2.5GB - SLOW hidden thinking tokens, avoid!
]

results = {}
for model, ctx in models_to_test:
    try:
        t_small, t_real, correct = test_model(model, ctx)
        results[f"{model}"] = (t_small, t_real, correct)
    except Exception as e:
        print(f"\n{model}: ERROR - {e}")
        unload_model(model)

print(f"\n{'='*60}")
print("MODEL SELECTION SUMMARY")
print(f"{'='*60}")
for model, (t_small, t_real, correct) in sorted(results.items(), key=lambda x: x[1][1]):
    status = "✅ USE THIS" if correct and t_real < 15 else "⚠️ SLOW" if correct else "❌ BAD"
    print(f"  {model}: {t_real:.1f}s {status}")


qwen2.5:3b (ctx=4096):
  Warm-up: 4.6s
  Small prompt:     3.02s
  Realistic prompt: 5.63s (✅ format)
  Output: [012_001]|||The high cost of green hydrogen remains a significant barrier to decarbonisation in the steel industry.
[012...

MODEL SELECTION SUMMARY
  qwen2.5:3b: 5.6s ✅ USE THIS


In [11]:
# =============================================================================
# TEST 2: QUALITY COMPARISON - Compare models on REAL data
# =============================================================================
# Synthetic prompts can confuse small models. Test on actual chunks instead.

import time
import subprocess

def unload_model(model):
    try:
        subprocess.run(["ollama", "stop", model], capture_output=True, timeout=10)
        time.sleep(2)
    except:
        pass

def test_real_extraction(model, company="012", year="2021"):
    """Run extraction on real data and compare results."""
    from nlp.rag_1 import RAGConfig, RAGPipeline

    unload_model(model)  # Clean slate

    config = RAGConfig(
        cache_dir="../cache",
        ollama_model=model,
        min_detector_score=0.7,
        max_chunks_per_group=7,
        llm_num_ctx=4096,
    )
    pipeline = RAGPipeline(config)
    pipeline.load_from_cache()

    # Time the extraction
    start = time.time()
    barriers, motivators = pipeline.extract_company_year(company, year)
    elapsed = time.time() - start

    print(f"\n{'='*60}")
    print(f"{model} on ({company}, {year})")
    print(f"{'='*60}")
    print(f"Time: {elapsed:.1f}s")
    print(f"Barriers: {len(barriers)}, Motivators: {len(motivators)}")

    print(f"\nBarriers extracted:")
    for cid, text in barriers[:8]:
        print(f"  [{cid}] {text[:70]}...")

    unload_model(model)
    return barriers, motivators, elapsed

# === COMPARE MODELS ON REAL DATA ===
# Uses smallest group: company 012, year 2021 (10 chunks)

models_to_compare = [
    "qwen2.5:3b",   # 1.9GB - no thinking mode, should be fast + good quality
    # "gemma3:1b",    # 815MB - fast but lower quality
    # "gemma3:4b",    # 3.3GB - best quality but slow (50% CPU)
]

results = {}
for model in models_to_compare:
    try:
        b, m, t = test_real_extraction(model)
        results[model] = (b, m, t)
    except Exception as e:
        print(f"{model}: ERROR - {e}")

# Summary
print(f"\n{'='*60}")
print("QUALITY COMPARISON SUMMARY")
print(f"{'='*60}")
for model, (barriers, motivators, elapsed) in sorted(results.items(), key=lambda x: x[1][2]):
    print(f"\n{model}:")
    print(f"  Time: {elapsed:.1f}s")
    print(f"  Barriers: {len(barriers)}, Motivators: {len(motivators)}")

# Check overlap between models
if len(results) >= 2:
    print(f"\n--- Overlap Analysis ---")
    model_names = list(results.keys())
    for i, m1 in enumerate(model_names):
        for m2 in model_names[i+1:]:
            b1 = set(text.lower()[:50] for _, text in results[m1][0])
            b2 = set(text.lower()[:50] for _, text in results[m2][0])
            overlap = len(b1 & b2)
            print(f"  {m1} ∩ {m2}: {overlap} barriers in common")

✓ RAG Pipeline initialized (GPU: NVIDIA T1200 Laptop GPU (3.9GB))
✓ Filtered 1048 chunks below detector_score=0.7 (14545 remaining)
('001', '2022'): 503 chunks
('001', '2023'): 487 chunks
('001', '2021'): 484 chunks
('015', '2024'): 344 chunks
('001', '2019'): 338 chunks
('001', '2024'): 336 chunks
('001', '2020'): 332 chunks
('003', '2024'): 279 chunks
('015', '2025'): 277 chunks
('001', '2013'): 262 chunks
✓ Loaded 14545 chunks from 15 companies (../cache)
Loading Ollama model: qwen2.5:3b

qwen2.5:3b on (012, 2021)
Time: 24.9s
Barriers: 1, Motivators: 0

Barriers extracted:
  [012_018] In order to achieve the climate objectives, in particular to initiate ...

QUALITY COMPARISON SUMMARY

qwen2.5:3b:
  Time: 24.9s
  Barriers: 1, Motivators: 0


In [1]:
from nlp import quick_start, RAGConfig, analyze_token_usage

# Configure RAG pipeline parameters here
rag_config = RAGConfig(
    cache_dir="../cache",
    use_bert_cache=True,
    output_folder="../out",

    # LLM settings
    ollama_model="qwen2.5:3b",
    ollama_base_url="http://localhost:11434",
    llm_temperature=0.0,
    llm_num_ctx=4096,  # Context window size

    # Extraction settings
    max_chunks_per_group=7,  # Limit chunks sent to LLM per company-year

    # BERT filtering (reduce chunk count by filtering low-confidence chunks)
    min_detector_score=0.7,  # 0.0 = no filter, 0.7 = keep only confident climate chunks
)

# Initialize pipeline and analyze token usage
pipeline = quick_start(config=rag_config)
stats = analyze_token_usage(pipeline)

# Auto-apply recommended max_chunks_per_group (optional)
rag_config.max_chunks_per_group = stats["recommendations"]["max_chunks_for_ctx"]
stats["recommendations"]["max_chunks_for_ctx"]

/home/as/code/ai/susteelaible/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ RAG Pipeline initialized (GPU: NVIDIA T1200 Laptop GPU (3.9GB))
✓ Filtered 1048 chunks below detector_score=0.7 (14545 remaining)
('001', '2022'): 503 chunks
('001', '2023'): 487 chunks
('001', '2021'): 484 chunks
('015', '2024'): 344 chunks
('001', '2019'): 338 chunks
('001', '2024'): 336 chunks
('001', '2020'): 332 chunks
('003', '2024'): 279 chunks
('015', '2025'): 277 chunks
('001', '2013'): 262 chunks
✓ Loaded 14545 chunks from 15 companies (../cache)
PROMPT OVERHEAD
  BARRIER_MAP_PROMPT:   ~151 tokens (606 chars)
  MOTIVATOR_MAP_PROMPT: ~151 tokens (606 chars)
  Using max as overhead: 151 tokens

CHUNK-LEVEL STATISTICS
Total chunks: 14545

Chunk size (characters):
  Min: 38, Max: 19629, Mean: 1753, Median: 1453

Chunk size (tokens, approx):
  Min: 9, Max: 4907, Mean: 438, Median: 363

GROUP-LEVEL STATISTICS (company-year)
Total groups: 132

Chunks per group:
  Min: 5, Max: 503, Mean: 110.2, Median: 86

Tokens per group (incl. prompt overhead):
  Min: 1219, Max: 231731, Mean: 48

9

In [2]:
# pipeline.get_companies()  # Returns ['001', '003', '015', ...]


# can save
#df_barriers, df_motivators = pipeline.extract_company_data("001")
#pipeline.save_company_tables("001", df_barriers, df_motivators)

barriers, motivators = pipeline.extract_company_year("012", "2021") #small




Loading Ollama model: qwen2.5:3b


In [3]:
print(f"Barriers: {len(barriers)}")
for chunk_id, text in barriers:
    print(f"  [{chunk_id}] {text[:1000]}...")

print(f"Motivators: {len(motivators)}")
for chunk_id, text in motivators:
    print(f"  [{chunk_id}] {text[:1000]}...")

Barriers: 0
Motivators: 0


In [7]:
%timeit
# pipeline.inspect_pipeline_status()
pipeline.extract_all_companies()


EXTRACTING ALL COMPANIES
Total groups: 132
LLM context: 4096 tokens

Extracting: ArcelorMittal (001)
Years: ['2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
Chunks: 3811 | Batches: 430 × 2 (barriers+motivators) = 860 LLM calls


  001:   8%|▊         | 1/13 [1:42:52<20:34:33, 6172.79s/it]

    ⚠️ chunk_id '001_528' not found in lookup
    ⚠️ chunk_id '001_999' not found in lookup
    ⚠️ chunk_id '001_0124' not found in lookup


  001:  38%|███▊      | 5/13 [4:39:59<6:16:18, 2822.28s/it] 

    ⚠️ chunk_id '001_003' not found in lookup


...

  001:  77%|███████▋  | 10/13 [12:08:08<4:44:06, 5682.31s/it]

    ⚠️ chunk_id '001_300' not found in lookup


  001:  85%|████████▍ | 11/13 [14:23:57<3:34:34, 6437.36s/it]

    ⚠️ chunk_id '015_154' not found in lookup
    ⚠️ chunk_id '015_076' not found in lookup
    ⚠️ chunk_id '015_633' not found in lookup


KeyboardInterrupt: 

# rag 2

In [ ]:
# from bertopic.representation import OpenAI,LlamaCPP
from nlp import TopicModelConfig, run_topic_modeling_pipeline

# Set True to ignore cached model/embeddings and retrain from scratch
FORCE_RETRAIN = True

config = TopicModelConfig(
    # Embedding model
    # embedding_model="sentence-transformers/all-mpnet-base-v2",
    embedding_model="BAAI/bge-small-en-v1.5",

    batch_size=64,

    # UMAP (dimensionality reduction)
    umap_n_neighbors=30,
    umap_n_components=15,
    umap_min_dist=0.05,
    umap_metric='cosine',
    umap_random_state=42,

    # HDBSCAN (clustering)
    hdbscan_min_cluster_size=10,  # Higher = fewer topics
    hdbscan_min_samples=2,        # Lower = less outliers
    hdbscan_metric='euclidean',
    hdbscan_cluster_selection_method='leaf',  # 'eom' (inclusive) or 'leaf' (tight, more cleanup required)

    # Vectorizer (c-TF-IDF)
    vectorizer_ngram_range=(1, 2),
    vectorizer_min_df=1,
    vectorizer_max_df=0.95,

    # Topic representation
    mmr_diversity=0.2, # 0 - pure relevance, redundant/simi word ... 1 - pure diverse. max diff word
    top_n_words=10,
    nr_topics=10,  # Set None for auto, or int to reduce post-hoc
    calculate_probabilities=True,

    # Outlier reduction (post-hoc)
    reduce_outliers=False,  # Reassign outliers to nearest topic
    reduce_outliers_strategy='embeddings',  # 'embeddings', 'c-tf-idf', or 'distributions'

    # Visualization UMAP (separate 2D projection)
    viz_umap_n_neighbors=10,
    viz_umap_n_components=2,
    viz_umap_min_dist=0.0,

    # LLM for topic labeling
    ollama_model="gemma3:4b",  # Fast + follows format. Avoid qwen3 (hidden thinking = slow)
    ollama_base_url="http://localhost:11434",
    llm_temperature=0.0,

    # Misc
    verbose=True,
    embeddings_cache_path=None,
)

results = run_topic_modeling_pipeline(
    data_folder="../out",
    output_folder="../out/topics5",
    config=config,
    force_retrain=FORCE_RETRAIN
)
# Access results
barriers_df = results['barriers']['df']
motivators_df = results['motivators']['df']

In [ ]:
motivators_df